# PDF vector log → LAS (pyLogDigitizer)

Digitize **vector** (or high-res colour) well-log PDFs using `pypdf` (BSD).
Shares `DepthMap`, `ScaleMap`, `las_export.resample` / `write_las` with the scan workflow.
Traced curves are dense connected points (no smoothing). LAS step default **0.5 ft** (option 0.25).
Nulls where the curve sits on a track scale limit (clipped).

Demo uses Utah FORGE 78B-32 CBL (public GDR). The zip is downloaded at runtime — not committed.


In [ ]:
from __future__ import annotations
import urllib.request, zipfile
from pathlib import Path
import numpy as np
from pylogdigitizer.pdf_vector import digitize_pdf
from pylogdigitizer.qc_compare import metrics, to_grid

DATA = Path("example_data/pdf_vector"); DATA.mkdir(parents=True, exist_ok=True)
URL = "https://gdr.openei.org/files/1330/78B-32_CBL_LOG.zip"
ZIP, PDF = DATA / "78B-32_CBL_LOG.zip", DATA / "UniversityofUtah_Forge78B-32_6Oct21_CBL.Pdf"
if not PDF.exists():
    if not ZIP.exists():
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as zf:
        for n in zf.namelist():
            if n.lower().endswith((".pdf", ".las")):
                zf.extract(n, DATA)
                (DATA / n).rename(DATA / Path(n).name)
print("PDF", PDF, "exists", PDF.exists())


In [ ]:
# Schlumberger colours on this plot: green GR, blue TT, black CBL
COLORS = {"ECGR_STGC": (0.0, 0.75, 0.0), "TT": (0.0, 0.0, 1.0), "CBL": (0.0, 0.0, 0.0)}
log = digitize_pdf(PDF, pages=[19, 29], step=0.5, mnemonic_colors=COLORS)
print("well:", log.well)
for t in log.traces:
    print(t.mnemonic, f"{t.depth.min():.0f}-{t.depth.max():.0f} ft", "pts", t.info.get("n_pts"), "unit", t.unit)
las_out = DATA / "forge78B_cbl_pages_digitized.las"
log.to_las(las_out)
print("wrote", las_out)


In [ ]:
import lasio
las_path = next(DATA.glob("*CBL*Main*.las"), None) or next(DATA.glob("*.las"), None)
alias = {"ECGR_STGC": "GR_STGC", "TT": "TT", "CBL": "CBL"}
if las_path and las_path != las_out:
    df = lasio.read(str(las_path)).df()
    for t in log.traces:
        m = alias.get(t.mnemonic)
        if not m or m not in df.columns: continue
        ref = df[m].replace(-999.25, np.nan).dropna()
        ok = np.isfinite(log.depth) & np.isfinite(log.curves[t.mnemonic])
        td, tv = log.depth[ok], log.curves[t.mnemonic][ok]
        print(t.mnemonic, "→", m, metrics(to_grid(ref.index.values, ref.values, td), tv))
else:
    print("no companion LAS for QC in", DATA)
